# OpenBayan Jupyter + Prefect Smoke Test

Run this notebook to verify that Jupyter, Prefect, CAMeL Tools, and the external Ollama host work together. The final cell runs a Prefect flow, so the run should also appear in the Prefect UI at http://localhost:4200.

In [1]:
from pathlib import Path
import json
import os
import sys
import urllib.request

import jupyterlab
import torch
from camel_tools.disambig.mle import MLEDisambiguator
from prefect import flow, get_run_logger, task

NOTEBOOK_ROOT = Path.cwd()
OLLAMA_URL = (os.environ.get("OLLAMA_URL") or os.environ.get("OLLAMA_HOST") or "").rstrip("/")

print("Notebook root:", NOTEBOOK_ROOT)
print("Prefect API URL:", os.environ.get("PREFECT_API_URL", "<not set>"))
print("Ollama URL set:", bool(OLLAMA_URL))

Notebook root: /app/notebooks
Prefect API URL: http://prefect-server:4200/api
Ollama URL set: True


## Prefect Tasks

These tasks can be edited and rerun directly in Jupyter during development.

In [2]:
@task(name="Check Notebook Environment")
def check_notebook_environment() -> dict:
    files = sorted(path.name for path in NOTEBOOK_ROOT.iterdir())
    return {
        "python": sys.version.split()[0],
        "cwd": str(NOTEBOOK_ROOT),
        "files": files,
        "prefect_api_url": os.environ.get("PREFECT_API_URL"),
    }


@task(name="Check Python Packages")
def check_python_packages() -> dict:
    return {
        "jupyterlab": jupyterlab.__version__,
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
    }


@task(name="Check CAMeL MLE")
def check_camel_mle(word: str = "كتاب") -> dict:
    disambig = MLEDisambiguator.pretrained()
    result = disambig.disambiguate([word])[0]
    return {
        "word": word,
        "lex": result.analyses[0].analysis.get("lex"),
    }

In [3]:
@task(name="List Ollama Models")
def list_ollama_models() -> list[str]:
    if not OLLAMA_URL:
        raise RuntimeError("Set OLLAMA_URL or OLLAMA_HOST before calling Ollama.")

    with urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=10) as response:
        data = json.load(response)

    return [model["name"] for model in data.get("models", [])]


@task(name="Generate Ollama Embedding")
def generate_ollama_embedding(text: str = "اختبار الاتصال") -> dict:
    if not OLLAMA_URL:
        raise RuntimeError("Set OLLAMA_URL or OLLAMA_HOST before calling Ollama.")

    embed_model = os.environ.get("OLLAMA_EMBED_MODEL", "mxbai-embed-large:latest")
    payload = json.dumps({"model": embed_model, "prompt": text}).encode("utf-8")
    request = urllib.request.Request(
        f"{OLLAMA_URL}/api/embeddings",
        data=payload,
        headers={"Content-Type": "application/json"},
    )

    with urllib.request.urlopen(request, timeout=30) as response:
        data = json.load(response)

    embedding = data.get("embedding", [])
    return {
        "model": embed_model,
        "dimensions": len(embedding),
        "first_values": embedding[:5],
    }

## Prefect Flow

Running this function creates a real Prefect flow run from inside the notebook.

In [4]:
@flow(name="OpenBayan Notebook Ollama Smoke Test")
def notebook_ollama_smoke_test() -> dict:
    logger = get_run_logger()

    environment = check_notebook_environment()
    packages = check_python_packages()
    camel = check_camel_mle()
    models = list_ollama_models()
    embedding = generate_ollama_embedding()

    summary = {
        "python": environment["python"],
        "notebook_root": environment["cwd"],
        "jupyterlab": packages["jupyterlab"],
        "torch": packages["torch"],
        "cuda_available": packages["cuda_available"],
        "camel_lex": camel["lex"],
        "ollama_model_count": len(models),
        "ollama_embedding_model": embedding["model"],
        "ollama_embedding_dimensions": embedding["dimensions"],
    }

    logger.info("Notebook smoke test summary: %s", summary)
    return summary

In [5]:
result = notebook_ollama_smoke_test()
result

03:46:37.762 | INFO    | Flow run 'towering-bulldog' - Beginning flow run 'towering-bulldog' for flow 'OpenBayan Notebook Ollama Smoke Test'

03:46:37.771 | INFO    | Flow run 'towering-bulldog' - View at http://prefect-server:4200/runs/flow-run/293ba142-06f0-4bce-a582-28a4ab8cd0e2

03:46:37.781 | INFO    | Task run 'Check Notebook Environment-2de' - Finished in state Completed()

03:46:37.786 | INFO    | Task run 'Check Python Packages-7da' - Finished in state Completed()

03:46:39.840 | INFO    | Task run 'Check CAMeL MLE-773' - Finished in state Completed()

03:46:39.932 | INFO    | Task run 'List Ollama Models-154' - Finished in state Completed()

03:46:40.831 | INFO    | Task run 'Generate Ollama Embedding-b59' - Finished in state Completed()

03:46:40.834 | INFO    | Flow run 'towering-bulldog' - Notebook smoke test summary: {'python': '3.12.13', 'notebook_root': '/app/notebooks', 'jupyterlab': '4.5.6', 'torch': '2.5.1+cpu', 'cuda_available': False, 'camel_lex': 'كِتاب', 'ollama_model_count': 25, 'ollama_embedding_model': 'mxbai-embed-large:latest', 'ollama_embedding_dimensions': 1024}

03:46:41.805 | INFO    | Flow run 'towering-bulldog' - Finished in state Completed()

{'python': '3.12.13',
 'notebook_root': '/app/notebooks',
 'jupyterlab': '4.5.6',
 'torch': '2.5.1+cpu',
 'cuda_available': False,
 'camel_lex': 'كِتاب',
 'ollama_model_count': 25,
 'ollama_embedding_model': 'mxbai-embed-large:latest',
 'ollama_embedding_dimensions': 1024}